# TheLook E-commerce — Data Profiling

**Purpose:** profile the data to inform marts modeling.

**Snapshot:** pinned Kaggle TheLook mirror (2024-01-21) → `scripts/prepare_data.sh` (requires `THELOOK_ZIP`) → `load.py` → `raw` → `dbt/models/staging` → `dbt_staging`.

**Run order:** `scripts/prepare_data.sh` → `python load.py` → `cd dbt && dbt build --profiles-dir .` → run this notebook.

In [1]:
import warnings
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

# DuckDB's .df() still uses DataFrame.drop(inplace=True); silence the upstream
# pandas 4 deprecation until the driver stops emitting it.
warnings.filterwarnings("ignore", message=".*inplace keyword in DataFrame.drop.*")

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

# Works whether the kernel starts in notebooks/ or the repo root.
candidates = [Path("warehouse.duckdb"), Path("../warehouse.duckdb")]
warehouse = next(path for path in candidates if path.exists())
con = duckdb.connect(warehouse, read_only=True)
con.execute("SET TimeZone = 'UTC'")


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


# table -> (grain, primary key)
GRAINS = {
    "stg_distribution_centers": ("one row per distribution center", "distribution_center_id"),
    "stg_products": ("one row per product", "product_id"),
    "stg_users": ("one row per user", "user_id"),
    "stg_orders": ("one row per order", "order_id"),
    "stg_order_items": ("one row per order item (line item)", "order_item_id"),
    "stg_inventory_items": ("one row per inventory unit", "inventory_item_id"),
    "stg_events": ("one row per clickstream event", "event_id"),
}

print(f"warehouse: {warehouse.resolve()}")
print(f"duckdb:    {duckdb.__version__}")


warehouse: /home/jeffer/thelook-analytics/warehouse.duckdb
duckdb:    1.5.4


## 1. Grains and keys

One row per order and one row per order item are separate grains. Joining the two before aggregating duplicates order-level measures across item rows. The full staging column list is in [Appendix A](#appendix-columns).


In [2]:
rows = []
for table, (grain, key) in GRAINS.items():
    result = q(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT {key}) AS distinct_keys,
            COUNT(*) - COUNT(DISTINCT {key}) AS duplicate_keys
        FROM dbt_staging.{table}
    """).iloc[0]
    rows.append({
        "table": table,
        "grain": grain,
        "key": key,
        "rows": int(result.rows),
        "distinct_keys": int(result.distinct_keys),
        "duplicate_keys": int(result.duplicate_keys),
    })

inventory = pd.DataFrame(rows)
display(inventory)


,table,grain,key,rows,distinct_keys,duplicate_keys
0,stg_distribution_centers,one row per distribution center,distribution_center_id,10,10,0
1,stg_products,one row per product,product_id,29120,29120,0
2,stg_users,one row per user,user_id,100000,100000,0
3,stg_orders,one row per order,order_id,125226,125226,0
4,stg_order_items,one row per order item (line item),order_item_id,181759,181759,0
5,stg_inventory_items,one row per inventory unit,inventory_item_id,490705,490705,0
6,stg_events,one row per clickstream event,event_id,2431963,2431963,0


### 1.1 Grain reconciliation

`stg_orders.item_count` declares how many line items belong to each order. It matches the actual count in `stg_order_items` for all 125,226 orders: no order is missing items, and every order has between one and four.


In [3]:
display(q("""
WITH per_order AS (
    SELECT
        o.order_id,
        o.item_count,
        COUNT(oi.order_item_id) AS actual_items
    FROM dbt_staging.stg_orders o
    LEFT JOIN dbt_staging.stg_order_items oi USING (order_id)
    GROUP BY 1, 2
)
SELECT
    COUNT(*) AS orders,
    COUNT(*) FILTER (WHERE actual_items = 0) AS orders_without_items,
    COUNT(*) FILTER (WHERE actual_items <> item_count) AS item_count_mismatches,
    MIN(item_count) AS min_declared_items,
    MAX(item_count) AS max_declared_items,
    CAST(SUM(item_count) AS BIGINT) AS declared_items,
    CAST(SUM(actual_items) AS BIGINT) AS actual_items
FROM per_order
"""))


,orders,orders_without_items,item_count_mismatches,min_declared_items,max_declared_items,declared_items,actual_items
0,125226,0,0,1,4,181759,181759


### 1.2 User attributes

`country` has 16 labels; `Deutschland` and `España` appear twice each.


In [4]:
display(q("""
    SELECT country, COUNT(*) AS users
    FROM dbt_staging.stg_users
    GROUP BY 1
    ORDER BY users DESC
"""))


,country,users
0,China,34150
1,United States,22522
2,Brasil,14507
3,South Korea,5316
4,France,4700
5,United Kingdom,4561
6,Germany,4155
7,Spain,4062
8,Japan,2438
9,Australia,2146


## 2. Time coverage

Stocking starts first (2018-11-08), and the other four timestamp columns start in early January 2019. Users stop on 2024-01-16 and orders on 2024-01-17, while stocking, item and event timestamps reach 2024-01-21. 2024 is therefore partial, and full-year comparisons should stop at 2023.


In [5]:
display(q("""
    SELECT 'stg_orders.ordered_at' AS column_name, MIN(ordered_at) AS min_at, MAX(ordered_at) AS max_at
    FROM dbt_staging.stg_orders
    UNION ALL
    SELECT 'stg_order_items.item_created_at', MIN(item_created_at), MAX(item_created_at)
    FROM dbt_staging.stg_order_items
    UNION ALL
    SELECT 'stg_users.registered_at', MIN(registered_at), MAX(registered_at)
    FROM dbt_staging.stg_users
    UNION ALL
    SELECT 'stg_events.event_at', MIN(event_at), MAX(event_at)
    FROM dbt_staging.stg_events
    UNION ALL
    SELECT 'stg_inventory_items.stocked_at', MIN(stocked_at), MAX(stocked_at)
    FROM dbt_staging.stg_inventory_items
    ORDER BY 1
"""))


,column_name,min_at,max_at
0,stg_events.event_at,2019-01-02 00:05:00+00:00,2024-01-21 18:02:23.533893+00:00
1,stg_inventory_items.stocked_at,2018-11-08 04:14:18+00:00,2024-01-21 03:29:15.939041+00:00
2,stg_order_items.item_created_at,2019-01-06 02:25:41+00:00,2024-01-21 18:02:23.533893+00:00
3,stg_orders.ordered_at,2019-01-06 05:30:00+00:00,2024-01-17 19:46:14.316147+00:00
4,stg_users.registered_at,2019-01-02 00:06:00+00:00,2024-01-16 19:46:14.316147+00:00


## 3. Revenue definitions

Gross revenue is the sum of `sale_price` over every line item. Only items on `complete` or `shipped` orders are counted as realized; the rest are pending (`processing`), cancelled, or returned. `shipped` counts as realized because those orders have no later delivery or return event in the snapshot. Gross exceeds realized revenue by 81%, or equivalently 45% of gross is not realized. Processing, cancelled and returned orders account for 20.1%, 14.9% and 10.0% of orders.


In [6]:
display(q("""
    WITH order_revenue AS (
        SELECT
            o.order_id,
            o.order_status,
            COUNT(oi.order_item_id) AS item_count,
            SUM(oi.sale_price) AS gross_revenue
        FROM dbt_staging.stg_orders o
        JOIN dbt_staging.stg_order_items oi USING (order_id)
        GROUP BY 1, 2
    )
    SELECT
        order_status,
        COUNT(*) AS orders,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS order_pct,
        CAST(SUM(item_count) AS BIGINT) AS items,
        ROUND(SUM(gross_revenue), 2) AS gross_revenue,
        ROUND(COALESCE(SUM(gross_revenue) FILTER (WHERE order_status IN ('complete', 'shipped')), 0), 2) AS realized_revenue
    FROM order_revenue
    GROUP BY 1
    ORDER BY orders DESC
"""))

display(q("""
    WITH order_revenue AS (
        SELECT
            o.order_id,
            SUM(oi.sale_price) AS gross_revenue,
            SUM(oi.sale_price) FILTER (WHERE o.order_status IN ('complete', 'shipped')) AS realized_revenue
        FROM dbt_staging.stg_orders o
        JOIN dbt_staging.stg_order_items oi USING (order_id)
        GROUP BY 1
    )
    SELECT
        ROUND(AVG(gross_revenue), 2) AS avg_gross_order_value_all_orders,
        ROUND(AVG(realized_revenue) FILTER (WHERE realized_revenue > 0), 2) AS avg_realized_order_value_realized_orders,
        ROUND(SUM(gross_revenue), 2) AS total_gross_revenue,
        ROUND(SUM(realized_revenue), 2) AS total_realized_revenue
    FROM order_revenue
"""))


,order_status,orders,order_pct,items,gross_revenue,realized_revenue
0,shipped,37577,30.01,54440,3256221.46,3256221.46
1,complete,31354,25.04,45609,2715783.74,2715783.74
2,processing,25156,20.09,36388,2155331.51,0.00
3,cancelled,18609,14.86,27090,1602475.45,0.00
4,returned,12530,10.01,18232,1097306.74,0.00


,avg_gross_order_value_all_orders,avg_realized_order_value_realized_orders,total_gross_revenue,total_realized_revenue
0,86.46,86.64,10827118.9,5972005.2


### 3.1 Price semantics

`sale_price` equals `stg_products.retail_price` for all 181,759 line items, and every value is positive ($0.02–$999.00). The snapshot has no discount field and no price history, so `sale_price` is simply the product's list price. The data cannot distinguish list price from a discounted transaction price because no discount is ever applied. Gross revenue is therefore retail merchandise value; realization is handled separately by order status.

`stg_products.unit_cost` supports margin analysis directly. No product has a negative margin; the minimum is $0.01, the median $20.06, and the average margin is 51.1% of retail price. If margin is out of scope, the marts need only the revenue measures and should say so.


In [7]:
display(q("""
    SELECT
        COUNT(*) AS items,
        COUNT(*) FILTER (WHERE oi.sale_price = p.retail_price) AS matches_product_retail_price,
        COUNT(*) FILTER (WHERE oi.sale_price <> p.retail_price) AS differs_from_product_retail_price,
        COUNT(*) FILTER (WHERE oi.sale_price <= 0) AS nonpositive_sale_price,
        MIN(oi.sale_price) AS min_sale_price,
        MAX(oi.sale_price) AS max_sale_price
    FROM dbt_staging.stg_order_items oi
    JOIN dbt_staging.stg_products p USING (product_id)
"""))

display(q("""
    SELECT
        COUNT(*) AS products,
        COUNT(*) FILTER (WHERE retail_price < unit_cost) AS negative_margin_products,
        ROUND(MIN(retail_price - unit_cost), 2) AS min_margin,
        ROUND(MEDIAN(retail_price - unit_cost), 2) AS median_margin,
        ROUND(AVG((retail_price - unit_cost) / NULLIF(retail_price, 0)) * 100, 2) AS avg_margin_pct
    FROM dbt_staging.stg_products
"""))


,items,matches_product_retail_price,differs_from_product_retail_price,nonpositive_sale_price,min_sale_price,max_sale_price
0,181759,181759,0,0,0.02,999.0


,products,negative_margin_products,min_margin,median_margin,avg_margin_pct
0,29120,0,0.01,20.06,51.08


## 4. Timestamps and lifecycle


### 4.1 Order lifecycle and status/timestamp matrix

`stg_orders.ordered_at` (unlike `stg_orders.item_created_at`) is consistent with the order lifecycle (`ordered_at <= shipped_at <= delivered_at <= returned_at`). The query below counts both out-of-order timestamps and lifecycle steps that skip a stage, such as a delivery without a shipment or a return without a delivery; all three counts are zero.


In [8]:
display(q("""
    SELECT
        COUNT(*) AS orders,
        COUNT(*) FILTER (WHERE shipped_at IS NOT NULL AND shipped_at < ordered_at) AS ship_before_order,
        COUNT(*) FILTER (WHERE delivered_at IS NOT NULL AND (shipped_at IS NULL OR delivered_at < shipped_at)) AS deliver_before_ship,
        COUNT(*) FILTER (WHERE returned_at IS NOT NULL AND (delivered_at IS NULL OR returned_at < delivered_at)) AS return_before_deliver
    FROM dbt_staging.stg_orders
"""))


,orders,ship_before_order,deliver_before_ship,return_before_deliver
0,125226,0,0,0


**Status/timestamp matrix.** `shipped` has a shipment timestamp but no delivery, `complete` has both but no return, and `returned` has all three. `processing` and `cancelled` have none. Item status matches the parent order for all 181,759 items, so there are no partial returns and order status alone is sufficient for revenue classification.


In [9]:
display(q("""
    SELECT
        order_status,
        COUNT(*) AS orders,
        COUNT(*) FILTER (WHERE shipped_at IS NOT NULL) AS has_shipped_at,
        COUNT(*) FILTER (WHERE delivered_at IS NOT NULL) AS has_delivered_at,
        COUNT(*) FILTER (WHERE returned_at IS NOT NULL) AS has_returned_at
    FROM dbt_staging.stg_orders
    GROUP BY 1
    ORDER BY orders DESC
"""))

display(q("""
    SELECT
        COUNT(*) AS order_items,
        COUNT(*) FILTER (WHERE oi.item_status = o.order_status) AS matching_item_status,
        COUNT(*) FILTER (WHERE oi.item_status IS DISTINCT FROM o.order_status) AS differing_item_status
    FROM dbt_staging.stg_order_items oi
    JOIN dbt_staging.stg_orders o USING (order_id)
"""))


,order_status,orders,has_shipped_at,has_delivered_at,has_returned_at
0,shipped,37577,37577,0,0
1,complete,31354,31354,31354,0
2,processing,25156,0,0,0
3,cancelled,18609,0,0,0
4,returned,12530,12530,12530,12530


,order_items,matching_item_status,differing_item_status
0,181759,181759,0


<a id="item-timestamps"></a>

### 4.2 Item timestamps vs the order's

`shipped_at`, `delivered_at` and `returned_at` are identical to the parent order's for every item, so they carry no item-level information. `item_created_at` is the exception: it equals the order timestamp for 11 of 181,759 items, and the difference ranges from −4.0 to +96.4 hours, with 43.5% of items timestamped after their order. It is the sale timestamp of the line item. The inventory unit referenced by `inventory_item_id` is unsold; the sold unit is recoverable by matching `(product_id, sold_at)` to `item_created_at`.


In [10]:
display(q("""
    SELECT item_column, matches_order, differs_from_order
    FROM (
        SELECT 1 AS ord, 'shipped_at' AS item_column,
               COUNT(*) FILTER (WHERE oi.shipped_at IS NOT DISTINCT FROM o.shipped_at) AS matches_order,
               COUNT(*) FILTER (WHERE oi.shipped_at IS DISTINCT FROM o.shipped_at) AS differs_from_order
        FROM dbt_staging.stg_order_items oi
        JOIN dbt_staging.stg_orders o USING (order_id)
        UNION ALL
        SELECT 2, 'delivered_at',
               COUNT(*) FILTER (WHERE oi.delivered_at IS NOT DISTINCT FROM o.delivered_at),
               COUNT(*) FILTER (WHERE oi.delivered_at IS DISTINCT FROM o.delivered_at)
        FROM dbt_staging.stg_order_items oi
        JOIN dbt_staging.stg_orders o USING (order_id)
        UNION ALL
        SELECT 3, 'returned_at',
               COUNT(*) FILTER (WHERE oi.returned_at IS NOT DISTINCT FROM o.returned_at),
               COUNT(*) FILTER (WHERE oi.returned_at IS DISTINCT FROM o.returned_at)
        FROM dbt_staging.stg_order_items oi
        JOIN dbt_staging.stg_orders o USING (order_id)
        UNION ALL
        SELECT 4, 'item_created_at',
               COUNT(*) FILTER (WHERE oi.item_created_at IS NOT DISTINCT FROM o.ordered_at),
               COUNT(*) FILTER (WHERE oi.item_created_at IS DISTINCT FROM o.ordered_at)
        FROM dbt_staging.stg_order_items oi
        JOIN dbt_staging.stg_orders o USING (order_id)
    )
    ORDER BY ord
"""))

display(q("""
    SELECT ROUND(MIN(h), 1) AS min_hours,
           ROUND(QUANTILE_CONT(h, 0.25), 1) AS q25_hours,
           ROUND(MEDIAN(h), 1) AS median_hours,
           ROUND(QUANTILE_CONT(h, 0.75), 1) AS q75_hours,
           ROUND(MAX(h), 1) AS max_hours
    FROM (
        SELECT DATE_DIFF('second', o.ordered_at, oi.item_created_at) / 3600.0 AS h
        FROM dbt_staging.stg_order_items oi
        JOIN dbt_staging.stg_orders o USING (order_id)
    )
"""))


,item_column,matches_order,differs_from_order
0,shipped_at,181759,0
1,delivered_at,181759,0
2,returned_at,181759,0
3,item_created_at,11,181748


,min_hours,q25_hours,median_hours,q75_hours,max_hours
0,-4.0,-2.2,-0.4,46.6,96.4


The event log is the only table with clickstream data. It has 2.4M events across 681,759 sessions, and its sessions fall into seven event-count shapes. The findings below cover the identity split, the stage sets, and the purchase-event grain.

<a id="coverage-and-identity"></a>

### 5.1 Coverage and identity

The clickstream contains 2.4M rows, of which 46.29% have no user. Sessions follow a small set of generated shapes (1, 2, 3, 5, 7, 10 or 13 events per session). The same 80,044 users appear under product, department, cart and purchase, 63,197 under home and none under cancel, so a user-level funnel is degenerate from product to purchase.


In [11]:
display(q("""
    SELECT
        event_type,
        COUNT(*) as events,
        COUNT(DISTINCT session_id) as sessions,
        COUNT(DISTINCT user_id) as users
    FROM dbt_staging.stg_events
    GROUP BY event_type
    ORDER BY events DESC
"""))

,event_type,events,sessions,users
0,product,845607,681759,80044
1,cart,595994,432146,80044
2,department,595323,431475,80044
3,purchase,181759,181759,80044
4,cancel,125568,125568,0
5,home,87712,87712,63197


In [12]:
display(q("""
    SELECT
        COUNT(*) AS events,
        COUNT(*) FILTER (WHERE user_id IS NULL) AS null_user,
        ROUND(100.0 * COUNT(*) FILTER (WHERE user_id IS NULL) / COUNT(*), 2) AS null_user_pct,
        COUNT(DISTINCT session_id) AS sessions,
        COUNT(DISTINCT user_id) AS users
    FROM dbt_staging.stg_events
"""))

display(q("""
    SELECT events_per_session, COUNT(*) AS sessions
    FROM (
        SELECT session_id, COUNT(*) AS events_per_session
        FROM dbt_staging.stg_events
        GROUP BY 1
    )
    GROUP BY 1
    ORDER BY 1
"""))


,events,null_user,null_user_pct,sessions,users
0,2431963,1125671,46.29,681759,80044


,events_per_session,sessions
0,1,124716
1,2,124897
2,3,250387
3,5,87712
4,7,49734
5,10,18825
6,13,25488


<a id="identity-finding"></a>

### 5.2 Identity is purchase-determined, not behavioral

Zero anonymous purchases is consistent with an account-required checkout, but that explains only one direction of the relationship. Every one of the 181,759 identified sessions contains a purchase: there are no identified browse-only or cart-abandonment sessions. It is not a first-order effect either — 45,048 users have multiple identified sessions (up to 14 per user), and every one of those sessions purchases. All 80,044 identified users registered before their first event and first order (median 256 days before the first order, minimum 1 day), so these are not accounts created at checkout. `user_id` therefore behaves as a purchase-attribution flag, not a login flag. The event log contains two disjoint generated populations — anonymous non-converting sessions and identified purchase journeys — so conversion is not identifiable.

This also explains the cart rate: carts total 432,146 sessions, split 250,387 anonymous and 181,759 identified, and the 42.1% cart-to-purchase rate is exactly the identified share of carts. The 125,568 `cancel` sessions are anonymous cart-only sessions, so `cancel` is cart abandonment, not order cancellation.


In [13]:
display(q("""
    WITH per_session AS (
        SELECT
            session_id,
            COUNT(*) AS events,
            COUNT(*) FILTER (WHERE user_id IS NOT NULL) AS identified_events,
            BOOL_OR(event_type = 'purchase') AS purchased,
            BOOL_OR(event_type = 'cart') AS carted,
            BOOL_OR(event_type = 'cancel') AS cancelled
        FROM dbt_staging.stg_events
        GROUP BY 1
    )
    SELECT
        CASE WHEN identified_events > 0 THEN 'identified' ELSE 'anonymous' END AS session_kind,
        COUNT(*) AS sessions,
        MIN(events) AS min_events,
        MAX(events) AS max_events,
        CAST(SUM(purchased::INT) AS BIGINT) AS purchase_sessions,
        CAST(SUM(carted::INT) AS BIGINT) AS cart_sessions,
        CAST(SUM(cancelled::INT) AS BIGINT) AS cancel_sessions,
        CAST(SUM((NOT purchased)::INT) AS BIGINT) AS nonpurchase_sessions
    FROM per_session
    GROUP BY 1
    ORDER BY 1
"""))

display(q("""
    WITH identified_sessions AS (
        SELECT user_id, COUNT(DISTINCT session_id) AS sessions
        FROM dbt_staging.stg_events
        WHERE user_id IS NOT NULL
        GROUP BY 1
    ),
    first_orders AS (
        SELECT user_id, MIN(ordered_at) AS first_order_at
        FROM dbt_staging.stg_orders
        GROUP BY 1
    )
    SELECT
        (SELECT COUNT(*) FROM identified_sessions WHERE sessions > 1) AS users_with_multiple_identified_sessions,
        (SELECT MAX(sessions) FROM identified_sessions) AS max_identified_sessions_per_user,
        (SELECT COUNT(*) FROM first_orders f JOIN dbt_staging.stg_users u USING (user_id)
         WHERE u.registered_at > f.first_order_at) AS users_registered_after_first_order,
        (SELECT CAST(MIN(DATE_DIFF('day', u.registered_at, f.first_order_at)) AS BIGINT)
         FROM first_orders f JOIN dbt_staging.stg_users u USING (user_id)) AS min_days_registered_before_first_order,
        (SELECT ROUND(MEDIAN(DATE_DIFF('day', u.registered_at, f.first_order_at)), 0)
         FROM first_orders f JOIN dbt_staging.stg_users u USING (user_id)) AS median_days_registered_before_first_order
"""))


,session_kind,sessions,min_events,max_events,purchase_sessions,cart_sessions,cancel_sessions,nonpurchase_sessions
0,anonymous,500000,1,3,0,250387,125568,500000
1,identified,181759,5,13,181759,181759,0,0


,users_with_multiple_identified_sessions,max_identified_sessions_per_user,users_registered_after_first_order,min_days_registered_before_first_order,median_days_registered_before_first_order
0,45048,14,0,1,256.0


### 5.3 Session stage nesting

Product is universal: all 681,759 sessions reach it. Cart implies product: 432,146 sessions reach cart (63.4%). Purchase implies cart: 181,759 sessions purchase (42.1% of carts, 26.7% of all sessions). Browse is separate because only 431,475 sessions have a home or department event.

The stages nest with zero violations, but that nesting is a consequence of how the sessions were generated, not evidence of a behavioral funnel (see [§5.2](#identity-finding)).


In [14]:
display(q("""
    WITH per_session AS (
        SELECT
            session_id,
            BOOL_OR(event_type IN ('home', 'department')) AS reached_browse,
            BOOL_OR(event_type = 'product') AS reached_product,
            BOOL_OR(event_type = 'cart') AS reached_cart,
            BOOL_OR(event_type = 'purchase') AS reached_purchase,
            BOOL_OR(event_type = 'cancel') AS saw_cancel
        FROM dbt_staging.stg_events
        GROUP BY session_id
    )
    -- *_without_* columns count sessions that skip a stage; all are zero
    SELECT
        COUNT(*) AS sessions,
        CAST(SUM(reached_browse::INT) AS BIGINT) AS reached_browse,
        CAST(SUM(reached_product::INT) AS BIGINT) AS reached_product,
        CAST(SUM(reached_cart::INT) AS BIGINT) AS reached_cart,
        CAST(SUM(reached_purchase::INT) AS BIGINT) AS reached_purchase,
        CAST(SUM((reached_purchase AND NOT reached_cart)::INT) AS BIGINT) AS purchase_without_cart,
        CAST(SUM((reached_cart AND NOT reached_product)::INT) AS BIGINT) AS cart_without_product,
        CAST(SUM((reached_purchase AND NOT reached_product)::INT) AS BIGINT) AS purchase_without_product,
        CAST(SUM((saw_cancel AND NOT reached_cart)::INT) AS BIGINT) AS cancel_without_cart
    FROM per_session
"""))

,sessions,reached_browse,reached_product,reached_cart,reached_purchase,purchase_without_cart,cart_without_product,purchase_without_product,cancel_without_cart
0,681759,431475,681759,432146,181759,0,0,0,0


### 5.4 Purchase events vs order items

The URI is always `/purchase` and carries no order key. Purchase events match order items in count, but there is no reliable key joining them: exact user-and-timestamp matches cover only about half, and a nearest-timestamp match within three minutes leaves 897 events ambiguous. The count parity, the session shapes in [§5.1](#coverage-and-identity), and the identity split in [§5.2](#identity-finding) are consistent with one purchase event per line item, but not proof.


In [15]:
display(q("""
SELECT
    (SELECT COUNT(*) FROM dbt_staging.stg_events WHERE event_type = 'purchase') AS purchase_events,
    (SELECT COUNT(*) FROM dbt_staging.stg_order_items) AS order_items,
    (SELECT COUNT(*) FROM dbt_staging.stg_orders) AS orders,
    (SELECT COUNT(DISTINCT session_id) FROM dbt_staging.stg_events WHERE event_type = 'purchase') AS purchase_sessions,
    (SELECT COUNT(*) FROM (
        SELECT session_id FROM dbt_staging.stg_events
        WHERE event_type = 'purchase' GROUP BY session_id HAVING COUNT(*) = 1
    )) AS sessions_with_one_purchase
"""))

,purchase_events,order_items,orders,purchase_sessions,sessions_with_one_purchase
0,181759,181759,125226,181759,181759


181,759 purchase events and 181,759 order items, against 125,226 orders (87,712 with one item, 24,867 with two, 6,275 with three). The parity is consistent with one purchase event per line item, but it is not proof and does not provide a join key: exact user-and-timestamp matches cover only about half of the items, and a ±180-second match leaves 897 purchase events with more than one candidate line item. Each purchase session holds exactly one purchase event, so multi-item orders span several sessions. Because identity is purchase-determined ([§5.2](#identity-finding)), these sessions are purchase journeys rather than a behavioral sample: a session's purchase flag counts items, not orders. Order counts and per-order revenue come from `stg_orders` and `stg_order_items`.


## 6. Findings and decisions

| Decision | Value | Consequence |
|---|---|---|
| Order date | `stg_orders.ordered_at` | earliest order-level timestamp; ordered_at <= shipped_at <= delivered_at <= returned_at |
| Item date | `item_created_at` = sale timestamp | retained as an item attribute; not used for order dates, cohorts or fulfilment |
| Status semantics | `order_status` drives every lifecycle timestamp | `item_status` matches the order for every item; no partial returns |
| Sale price | `sale_price = retail_price` for all items | list-price measure; no discount data, so a discounted transaction price is not observable; realization is status-based |
| Gross revenue | sum of `sale_price` over all items | `gross_revenue` |
| Realized revenue | status `complete` or `shipped` | `realized_revenue`; pending = `processing`, lost = `cancelled` + `returned` |
| Inventory | not a mart input; `stg_products` is canonical for cost and attributes | `order_items.inventory_item_id` points at unsold units; `(product_id, sold_at)` recovers the sold unit if sell-through is needed |
| Events | session-level only; identity is purchase-determined | no conversion funnel; `user_id` is a purchase-attribution flag, not login status |
| Country labels | normalize `Deutschland`/`España` | required before a country dimension; otherwise treat geography as out of scope |
| Time coverage | 2019-01 → 2024-01 (2024 partial) | 2019–2023 for full-year comparisons |


<a id="appendix-columns"></a>

## Appendix A. Staging column catalogue

In [16]:
display(q("""
    SELECT table_name, ordinal_position, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'dbt_staging'
    ORDER BY table_name, ordinal_position
"""))


,table_name,ordinal_position,column_name,data_type
0,stg_distribution_centers,1,distribution_center_id,BIGINT
1,stg_distribution_centers,2,distribution_center_name,VARCHAR
2,stg_distribution_centers,3,latitude,DOUBLE
3,stg_distribution_centers,4,longitude,DOUBLE
4,stg_events,1,event_id,BIGINT
5,stg_events,2,sequence_number,INTEGER
6,stg_events,3,event_at,TIMESTAMP WITH TIME ZONE
7,stg_events,4,user_id,BIGINT
8,stg_events,5,session_id,VARCHAR
9,stg_events,6,ip_address,VARCHAR
